# 02 · Cache v2 (0.5 mm/px, ventana 160 mm, orientación canónica)

Construye `/workspace/data/cache/v2_05mm/<study>/<series>.npy` (`uint8 [cortes, 320, 320]`) solo para las series con slot
asignado en `meta_v2` y todas las de test. Diseño en `docs/pasos/paso3_entrada_v2.md`.

- Píxeles y normalización con `read_series` de `src/knee/dicom.py` (slope/intercept, MONOCHROME1, p1–p99 por serie,
  cortes ordenados por posición proyectada `IPP·normal`, no por `InstanceNumber`).
- Orientación canónica (transpone/voltea según `ImageOrientationPatient`): filas S→I en sagital y coronal, A→P en axial;
  columnas A→P en sagital, derecha→izquierda del paciente en coronal y axial. Cortes: sagital R→L (+x), coronal A→P (+y),
  axial S→I (−z).
- Remuestreo en el plano a 0.5 mm/px y ventana de 320 px centrada en el centro del estudio (`study_cx/cy/cz` proyectado
  a los píxeles de la serie; si cae fuera de la imagen o falta, centro de la imagen). Relleno con ceros.
- Sin espejar: `index.csv` guarda `side` (= `study_side`) y `mirror_axis` (`slices` en sagital, `cols` en coronal/axial);
  el espejado se hace al leer (`KneeDataset(mirror=True)`).

Primero `SMOKE = True` (50 estudios a `v2_05mm_smoke/`, QA visual y estimación). La construcción completa es `SMOKE = False`.

In [ ]:
# CONFIG
EXP = "exp009_preparacion_v2"
RAW_ROOT = "/workspace/data/raw"
META_DIR = "/workspace/data/cache/meta_v2"
CACHE_VERSION = "v2_05mm"
SMOKE = True
SMOKE_STUDIES = 50
SPACING_MM = 0.5
WINDOW_PX = 320
QA_CROP_MM = 130
QA_PER_SLOT = 12
N_WORKERS = 8
SEED = 42

In [ ]:
from multiprocessing import Pool
from pathlib import Path
import json
import os
import re
import sys
import time

import cv2
import numpy as np
import pandas as pd
import pydicom

repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "knee").exists())
sys.path.insert(0, str(repo))
from src.knee.dicom import read_series

raw = Path(RAW_ROOT)
cache = Path("/workspace/data/cache") / (CACHE_VERSION + ("_smoke" if SMOKE else ""))
cache.mkdir(parents=True, exist_ok=True)
results = repo / "results" / EXP
print(f"Salida: {cache}")

## Series a construir: las que tienen slot + todas las de test

In [ ]:
series = pd.read_csv(Path(META_DIR) / "series_meta_v2.csv")
studies = pd.read_csv(Path(META_DIR) / "study_meta_v2.csv")

slot_cols = sorted(c for c in studies.columns if re.fullmatch(r"slot\d_\w+", c) and not c.endswith("_fallback"))
slot_of = pd.concat([
    studies[["study", col]].dropna().rename(columns={col: "series"}).assign(slot=int(col[4]))
    for col in slot_cols
])
assert not slot_of.series.duplicated().any(), "una serie en dos slots"

series = series.merge(slot_of[["series", "slot"]], on="series", how="left")
series = series.merge(studies[["study", "study_side", "study_cx", "study_cy", "study_cz"]], on="study", how="left")
jobs_all = series[series.slot.notna() | (series.split == "test")].reset_index(drop=True)
print(f"Series a construir (completo): {len(jobs_all):,} de {len(series):,} · estudios {jobs_all.study.nunique():,}")
print(jobs_all.groupby("split").agg(estudios=("study", "nunique"), series=("series", "count")))

if SMOKE:
    pick = jobs_all[jobs_all.split == "train"].study.drop_duplicates().sample(SMOKE_STUDIES, random_state=SEED)
    jobs = jobs_all[jobs_all.study.isin(pick)].reset_index(drop=True)
else:
    jobs = jobs_all
print(f"Esta corrida: {len(jobs):,} series de {jobs.study.nunique()} estudios")

## Geometría y construcción por serie

`geometry` lee solo la cabecera del primer archivo (IOP, IPP, PixelSpacing) y decide transposición, volteos, inversión
del orden de cortes y el centro en píxeles remuestreados. `build_one` es reanudable (si el `.npy` existe no relee
píxeles) y escribe a un temporal + `os.replace` para no dejar archivos a medias.

In [ ]:
TARGETS = {  # (dirección de filas, dirección de columnas) en LPS
    "Sagittal": (np.array([0, 0, -1.0]), np.array([0, 1.0, 0])),
    "Coronal": (np.array([0, 0, -1.0]), np.array([1.0, 0, 0])),
    "Axial": (np.array([0, 1.0, 0]), np.array([1.0, 0, 0])),
}
SLICE_ORDER = {"Sagittal": (0, 1), "Coronal": (1, 1), "Axial": (2, -1)}  # (eje LPS, signo): R→L, A→P, S→I


def plane_of(rec):
    if rec["plane"] in TARGETS:
        return rec["plane"]
    return rec["Anatomical_Plane"]  # oblicuas/unknown (solo test): plano del csv


def geometry(series_dir, plane, center):
    ds = pydicom.dcmread(sorted(series_dir.glob("*.dcm"))[0], stop_before_pixels=True)
    iop = np.asarray(ds.ImageOrientationPatient, dtype=float)
    r, c = iop[:3], iop[3:]  # r: columnas crecientes, c: filas crecientes
    ps_row, ps_col = (float(v) for v in ds.PixelSpacing)
    ipp = np.asarray(ds.ImagePositionPatient, dtype=float)
    rows, cols = int(ds.Rows), int(ds.Columns)
    t_row, t_col = TARGETS[plane]
    transposed = abs(r @ t_col) < abs(c @ t_col)
    if transposed:
        a_row, a_col, sp_row, sp_col, h, w = r, c, ps_col, ps_row, cols, rows
    else:
        a_row, a_col, sp_row, sp_col, h, w = c, r, ps_row, ps_col, rows, cols
    flip_rows, flip_cols = bool(a_row @ t_row < 0), bool(a_col @ t_col < 0)
    axis, sign = SLICE_ORDER[plane]
    reverse = bool(np.cross(r, c)[axis] * sign < 0)  # read_series ordena ascendente por IPP·(r×c)

    scale_r, scale_c = sp_row / SPACING_MM, sp_col / SPACING_MM
    h2, w2 = max(1, round(h * scale_r)), max(1, round(w * scale_c))
    src = "image"
    ci, cj = (h2 - 1) / 2, (w2 - 1) / 2
    if np.all(np.isfinite(center)):
        d = center - ipp
        i, j = d @ a_row / sp_row, d @ a_col / sp_col
        i, j = (h - 1 - i if flip_rows else i), (w - 1 - j if flip_cols else j)
        ii, jj = (i + 0.5) * scale_r - 0.5, (j + 0.5) * scale_c - 0.5
        if 0 <= ii < h2 and 0 <= jj < w2:
            ci, cj, src = ii, jj, "study"
    return {"shape": (rows, cols), "transposed": bool(transposed), "flip_rows": flip_rows, "flip_cols": flip_cols,
            "reversed": reverse, "size": (h2, w2), "center": (ci, cj), "center_src": src,
            "native_mm": round(min(ps_row, ps_col), 4)}


def resample(volume, g):
    if g["transposed"]:
        volume = volume.transpose(0, 2, 1)
    if g["flip_rows"]:
        volume = volume[:, ::-1]
    if g["flip_cols"]:
        volume = volume[:, :, ::-1]
    if g["reversed"]:
        volume = volume[::-1]
    h2, w2 = g["size"]
    interp = cv2.INTER_AREA if h2 < volume.shape[1] else cv2.INTER_LINEAR
    out = np.zeros((len(volume), WINDOW_PX, WINDOW_PX), dtype=np.uint8)
    r0 = int(round(g["center"][0])) - WINDOW_PX // 2
    c0 = int(round(g["center"][1])) - WINDOW_PX // 2
    rs, re_ = max(r0, 0), min(r0 + WINDOW_PX, h2)
    cs, ce = max(c0, 0), min(c0 + WINDOW_PX, w2)
    for k, image in enumerate(volume):
        image = cv2.resize(np.ascontiguousarray(image), (w2, h2), interpolation=interp)
        out[k, rs - r0:re_ - r0, cs - c0:ce - c0] = image[rs:re_, cs:ce]
    return out


def build_one(rec):
    study, series_uid = rec["study"], rec["series"]
    plane = plane_of(rec)
    output = cache / study / f"{series_uid}.npy"
    split_dir = "train_series" if rec["split"] == "train" else "test_series"
    base = {"study": study, "series": series_uid, "plane": plane, "fluid": int(rec["Fluid_Sensitive"]),
            "fat": int(rec["Fat_Suppression"]), "path": str(output), "n_slices": 0, "error": "",
            "split": rec["split"], "slot": rec["slot"], "weighting": rec["weighting"],
            "side": rec["study_side"] if isinstance(rec["study_side"], str) else "unknown",
            "spacing_mm": SPACING_MM, "mirror_axis": "slices" if plane == "Sagittal" else "cols", "seconds": 0.0}
    start = time.time()
    try:
        g = geometry(raw / split_dir / study / series_uid, plane,
                     np.array([rec["study_cx"], rec["study_cy"], rec["study_cz"]], dtype=float))
        base.update({"center_px": f"{g['center'][0]:.1f},{g['center'][1]:.1f}", "center_src": g["center_src"],
                     "resampled_hw": f"{g['size'][0]}x{g['size'][1]}", "native_mm": g["native_mm"],
                     "transposed": g["transposed"], "flip_rows": g["flip_rows"], "flip_cols": g["flip_cols"],
                     "reversed": g["reversed"]})
        if output.exists():
            base["n_slices"] = int(np.load(output, mmap_mode="r").shape[0])
            return base
        volume = read_series(raw / split_dir / study / series_uid)
        if volume.shape[1:] != g["shape"]:
            raise ValueError(f"forma {volume.shape[1:]} != cabecera {g['shape']}")
        out = resample(volume, g)
        output.parent.mkdir(parents=True, exist_ok=True)
        tmp = output.with_name(output.name + f".tmp{os.getpid()}.npy")
        np.save(tmp, out)
        os.replace(tmp, output)
        base.update({"n_slices": len(out), "seconds": round(time.time() - start, 2)})
    except Exception as exc:
        base.update({"path": "", "n_slices": 0, "error": f"{type(exc).__name__}: {exc}"})
    return base

## Construcción (pool por serie, reanudable)

In [ ]:
start = time.time()
rows = []
with Pool(N_WORKERS) as pool:
    for row in pool.imap_unordered(build_one, jobs.to_dict("records"), chunksize=1):
        rows.append(row)
        if len(rows) % 500 == 0:
            print(f"{len(rows):,}/{len(jobs):,} series · {time.time() - start:.0f} s", flush=True)
wall = time.time() - start

index = pd.DataFrame(rows).sort_values(["study", "slot", "series"]).reset_index(drop=True)
index.to_csv(cache / "index.csv", index=False)
errors = index[index.error != ""]
if len(errors):
    errors.to_csv(cache / "errors.csv", index=False)
print(f"{len(index) - len(errors):,}/{len(index):,} series OK · {len(errors)} errores · {wall / 60:.1f} min")
display(errors.head())
display(index.groupby("plane").agg(series=("series", "count"), cortes=("n_slices", "sum"),
                                   centro_estudio=("center_src", lambda s: (s == "study").mean()),
                                   transpuestas=("transposed", "mean"), invertidas=("reversed", "mean")))

## Estimación del cache completo (extrapolación lineal del smoke)

In [ ]:
ok = index[index.error == ""]
size_bytes = sum((Path(p).stat().st_size for p in ok.path))
per_series_bytes = size_bytes / len(ok)
n_full = len(jobs_all)
estimate = {
    "workers": N_WORKERS, "series_run": len(index), "wall_min": round(wall / 60, 2),
    "sec_per_series_wall": round(wall / len(index), 3),
    "mean_sec_per_series_cpu": round(float(ok.seconds.mean()), 3),
    "size_gb_run": round(size_bytes / 1e9, 3), "mb_per_series": round(per_series_bytes / 1e6, 2),
    "series_full": n_full,
    "est_full_hours": round(wall / len(index) * n_full / 3600, 2),
    "est_full_gb": round(per_series_bytes * n_full / 1e9, 1),
}
print(json.dumps(estimate, indent=1))

## QA visual

Por slot, 12 estudios al azar (mitad `L`, mitad `R` si hay): corte central tal como está guardado (sin espejar), con el
recorte de 130 mm marcado en rojo. Debe verse la rodilla centrada y la misma orientación en todas las filas.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

qa_dir = results / "qa_v2"
qa_dir.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(SEED)
half = QA_CROP_MM / SPACING_MM / 2
for slot in sorted(ok.slot.dropna().unique()):
    frame = ok[ok.slot == slot]
    picks = []
    for side in ["L", "R", "unknown"]:
        sub = frame[frame.side == side]
        picks.append(sub.sample(min(len(sub), QA_PER_SLOT // 2), random_state=SEED))
    pick = pd.concat(picks).head(QA_PER_SLOT)
    fig, axes = plt.subplots(3, 4, figsize=(16, 12.5))
    for ax in axes.flat:
        ax.axis("off")
    for ax, (_, row) in zip(axes.flat, pick.iterrows()):
        volume = np.load(row.path, mmap_mode="r")
        ax.imshow(volume[len(volume) // 2], cmap="gray", vmin=0, vmax=255)
        c = WINDOW_PX / 2
        ax.add_patch(plt.Rectangle((c - half, c - half), 2 * half, 2 * half, fill=False, ec="red", lw=1))
        ax.set_title(f"{row.study[-6:]} · {row.side} · {row.plane[:3]} {row.weighting} · {row.center_src}", fontsize=9)
    name = slot_cols[int(slot) - 1]
    fig.suptitle(f"{name} · filas/columnas: " + {"Sagittal": "S→I / A→P", "Coronal": "S→I / R→L",
                                                "Axial": "A→P / R→L"}[pick.plane.iloc[0]], fontsize=13)
    fig.tight_layout()
    fig.savefig(qa_dir / f"{name}.png", dpi=70)
    plt.close(fig)
print(sorted(p.name for p in qa_dir.glob("*.png")))

## Construcción completa

Con `SMOKE = False` (no borra el smoke). Reanudable: si se corta, se relanza igual y salta las series ya escritas.

```bash
cd /workspace/users/rodrigo/experiments/exp009_preparacion_v2 && nohup /workspace/venv/bin/python -m papermill \
  02_cache_v2.ipynb ../../results/exp009_preparacion_v2/02_cache_v2_full.ipynb -k rsna --log-output -p SMOKE False \
  > ../../results/exp009_preparacion_v2/02_cache_v2_full.log 2>&1 &
```